<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_18_functions_first_class/note_lesson_18_functions_first_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 18 — Функції як об'єкти першого класу: сервіс доставки росте

Карантин минув, а доставка «Смачно + Таксі» лишилася. Щотижня — нове: команда `drivers`, промокоди `SUMMER10`, `STUDENT15`, `NIGHT5`, сортування доставок, повідомлення в SMS і Telegram. У коді модуля 1 кожне таке прохання — ще один `elif` чи ще одна функція-близнюк.

Одна ідея розв'язує все: **функція — такий самий об'єкт, як число чи рядок**. Її можна покласти у змінну, у словник, передати в іншу функцію, повернути з функції.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія — у книзі: [Урок 18. Функції як об'єкти першого класу](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_18/).

## 🔁 Пригадай (без підглядання)

1. Що отримує і що повертає декоратор з уроку 9?
2. Що робить `max(revenue, key=revenue.get)`?
3. Як `app.py` фінального проєкту обирав команду?

<details>
<summary>Відповіді</summary>

1. Отримує функцію, повертає нову функцію-обгортку.
2. Повертає ключ з найбільшим значенням: `max` викликає `revenue.get` для кожного ключа.
3. Ланцюжком `if command == ...`.

</details>

## 1. Функція — це об'єкт

**Прогноз:** що надрукують три рядки?

In [ ]:
def fee_for(district):
    """Вартість доставки за районом."""
    return {"Поділ": 60, "Оболонь": 80}.get(district, 100)


print(type(fee_for))
print(fee_for.__name__)
print(fee_for("Поділ"))

<details>
<summary>Відповідь</summary>

`<class 'function'>`, `fee_for`, `60`. Без дужок — сам об'єкт-функція, з дужками — результат виклику.

</details>

In [ ]:
delivery_fee = fee_for
print(delivery_fee("Оболонь"), delivery_fee is fee_for)


bills = [540.0, 320.0, 980.0]
for summary in [len, sum, max]:
    print(summary.__name__, summary(bills))

## 2. Словник команд замість ланцюжка `if`

Нова команда — функція і **один рядок** у словнику. `run` не змінюється.

In [ ]:
def cmd_report(orders):
    return f"Замовлень: {len(orders)}, виторг: {sum(orders):.2f} грн"


def cmd_top(orders):
    return f"Найбільший чек: {max(orders):.2f} грн"


COMMANDS = {
    "report": cmd_report,
    "top": cmd_top,
}


def run(command, orders):
    handler = COMMANDS.get(command)
    if handler is None:
        return f"Помилка: невідома команда {command}. Є: {', '.join(COMMANDS)}"
    return handler(orders)


orders = [540.0, 320.0, 980.0]
print(run("report", orders))
print(run("top", orders))
print(run("fly", orders))

## 🛠 Вправа 1. Команда `drivers`

Функція `cmd_drivers(deliveries)` повертає водіїв за виторгом від більшого до меншого: `"D-3 500, D-1 180, D-2 150"`. Додай її в `COMMANDS` — `run` не змінюй.

In [ ]:
from typing import NamedTuple


class Delivery(NamedTuple):
    order_id: int
    district: str
    fare: int
    driver: str


deliveries = [
    Delivery(1, "Оболонь", 230, "D-3"),
    Delivery(2, "Поділ", 180, "D-1"),
    Delivery(3, "Оболонь", 270, "D-3"),
    Delivery(4, "Печерськ", 150, "D-2"),
]


def by_fare(delivery):
    return delivery.fare


print([d.order_id for d in sorted(deliveries, key=by_fare)])
print(max(deliveries, key=by_fare).order_id)

In [ ]:
from collections import Counter


def cmd_drivers(deliveries):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    totals = Counter()
    for delivery in deliveries:
        totals[delivery.driver] += delivery.fare
    names = sorted(totals, key=totals.get, reverse=True)
    return ", ".join(f"{name} {totals[name]}" for name in names)
    # END SOLUTION


# YOUR CODE HERE — додай команду в COMMANDS
# BEGIN SOLUTION
COMMANDS["drivers"] = cmd_drivers
# END SOLUTION

print(run("drivers", deliveries))
assert run("drivers", deliveries) == "D-3 500, D-1 180, D-2 150"
assert "drivers" in run("fly", deliveries)
print("✅ Вправа 1 пройдена")

## 3. Функція як аргумент: `key=`

**Прогноз:** у якому порядку `sorted` поверне доставки з ключем «район, потім вартість»?

In [ ]:
def by_district_then_fare(delivery):
    return (delivery.district, delivery.fare)


for d in sorted(deliveries, key=by_district_then_fare):
    print(d.district, d.fare)

<details>
<summary>Відповідь</summary>

Оболонь 230, Оболонь 270, Печерськ 150, Поділ 180 — спершу за районом (за абеткою), у межах району — за вартістю.

</details>

## 4. Callback: як повідомити — вирішує той, хто викликає

In [ ]:
def dispatch(delivery, notify):
    """Відправляє водія і повідомляє клієнта через notify(текст)."""
    notify(f"Замовлення №{delivery.order_id}: водій {delivery.driver} виїхав, {delivery.fare} грн")


def notify_sms(text):
    print("SMS:", text)


def notify_telegram(text):
    print("Telegram:", text)


dispatch(deliveries[0], notify_sms)
dispatch(deliveries[1], notify_telegram)


sent = []
dispatch(deliveries[2], sent.append)
print(sent)

## 5. Фабрика функцій і замикання

`make_discount(percent)` створює функцію-знижку, яка **пам'ятає** свій `percent`.

In [ ]:
def make_discount(percent):
    """Фабрика: повертає функцію, що знижує чек на percent відсотків."""
    def apply(bill):
        return round(bill * (100 - percent) / 100, 2)
    return apply


summer10 = make_discount(10)
student15 = make_discount(15)
print(summer10(540.0), student15(540.0))
print(summer10.__name__)


PROMO = {
    "SUMMER10": make_discount(10),
    "STUDENT15": make_discount(15),
    "NIGHT5": make_discount(5),
}

print(PROMO["NIGHT5"](980.0))

### `nonlocal`

**Прогноз:** що надрукує `print(other(), new_order_id())` після трьох викликів `new_order_id()`?

In [ ]:
def make_counter(start=0):
    count = start

    def next_id():
        nonlocal count
        count += 1
        return count

    return next_id


new_order_id = make_counter()
print(new_order_id(), new_order_id(), new_order_id())

other = make_counter(100)
print(other(), new_order_id())

<details>
<summary>Відповідь</summary>

`101 4`: кожен виклик `make_counter` створює **свій** лічильник, вони не заважають один одному.

</details>

### Пастка: функції в циклі

**Прогноз:** що надрукує клітинка? Потім виправ через фабрику.

In [ ]:
fees = [lambda bill: bill + fee for fee in (60, 80, 100)]
print([f(500) for f in fees])

## 🛠 Вправа 2. Виправ пастку фабрикою

Напиши `make_fee_adder(fee)`, що повертає функцію `bill -> bill + fee`, і збери `fees` через неї.

In [ ]:
def make_fee_adder(fee):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def add(bill):
        return bill + fee
    return add
    # END SOLUTION


fees = [make_fee_adder(fee) for fee in (60, 80, 100)]
print([f(500) for f in fees])
assert [f(500) for f in fees] == [560, 580, 600]
print("✅ Вправа 2 пройдена")

## 6. `lambda`, `map`, `filter`

`map` і `filter` ліниві, як генератори. Comprehension зазвичай читається легше.

In [ ]:
fares = [230, 180, 270, 150]
with_fee = map(lambda fare: fare + 20, fares)
print(type(with_fee).__name__)
print(list(with_fee))
print(list(filter(lambda fare: fare >= 200, fares)))

print([fare + 20 for fare in fares])
print([fare for fare in fares if fare >= 200])

## 7. Конвеєр з функцій

**Прогноз:** чому два конвеєри з тими самими кроками дають різну ціну?

In [ ]:
def add_fee(bill):
    return bill + 80


def to_uah(bill):
    return round(bill)


def run_pipeline(value, steps):
    for step in steps:
        value = step(value)
    return value


checkout = [PROMO["SUMMER10"], add_fee, to_uah]
print(run_pipeline(540.0, checkout))
print(run_pipeline(540.0, [add_fee, PROMO["SUMMER10"], to_uah]))

<details>
<summary>Відповідь</summary>

`566` і `558`: у другому знижку отримала й доставка. Порядок кроків — бізнес-правило.

</details>

### Кроки мають бути чистими

**Прогноз:** що буде в `cart` після кожного виклику?

In [ ]:
def add_packaging_bad(items):
    items.append("пакет")
    return items


def add_packaging(items):
    return items + ["пакет"]


cart = ["борщ", "вареники"]
print(add_packaging(cart), cart)
print(add_packaging_bad(cart), cart)

<details>
<summary>Відповідь</summary>

Чиста `add_packaging` повертає новий список і не змінює `cart`. Нечиста дописує «пакет» у сам `cart`.

</details>

## 8. Розібраний приклад: оформлення замовлення

In [ ]:
FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}


def make_fee(district):
    """Фабрика: крок конвеєра, що додає вартість доставки в район."""
    if district not in FEES:
        raise ValueError(f"не доставляємо в район {district}")
    fee = FEES[district]

    def add(bill):
        return bill + fee
    return add


def checkout_steps(promo_code, district):
    """Кроки для замовлення: знижка (якщо є промокод) → доставка → гривні."""
    steps = []
    if promo_code:
        if promo_code not in PROMO:
            raise ValueError(f"промокод {promo_code} не дійсний")
        steps.append(PROMO[promo_code])
    steps.append(make_fee(district))
    steps.append(to_uah)
    return steps


def checkout(bill, promo_code, district):
    return run_pipeline(bill, checkout_steps(promo_code, district))


CHECKOUT_CASES = [(540.0, "SUMMER10", "Оболонь"), (320.0, None, "Поділ"),
                  (980.0, "STUDENT15", "Печерськ"), (540.0, "FREE100", "Поділ"),
                  (540.0, None, "Троєщина")]

for bill, code, district in CHECKOUT_CASES:
    try:
        print(bill, code, district, "->", checkout(bill, code, district))
    except ValueError as error:
        print(bill, code, district, "-> помилка:", error)

## 🛠 Вправа 3. Промокод з обмеженням

`make_limited_discount(percent, uses)` — як `make_discount`, але після `uses` застосувань `apply` піднімає `ValueError("промокод вичерпано")`. Лічильник — у замиканні, через `nonlocal`.

In [ ]:
def make_limited_discount(percent, uses):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    left = uses

    def apply(bill):
        nonlocal left
        if left == 0:
            raise ValueError("промокод вичерпано")
        left -= 1
        return round(bill * (100 - percent) / 100, 2)
    return apply
    # END SOLUTION


lucky = make_limited_discount(20, 3)
assert [lucky(500.0), lucky(500.0), lucky(250.0)] == [400.0, 400.0, 200.0]
try:
    lucky(500.0)
except ValueError as error:
    assert str(error) == "промокод вичерпано"
else:
    raise AssertionError("четвертий виклик мав підняти ValueError")
other = make_limited_discount(20, 1)
assert other(100.0) == 80.0, "окремий промокод — окремий лічильник"
PROMO["LUCKY20"] = make_limited_discount(20, 3)
assert checkout(540.0, "LUCKY20", "Поділ") == 492
print("✅ Вправа 3 пройдена")

## 🐛 Вправа 4. Знайди помилку

Кожна клітинка — справжня помилка початківця. Розкоментуй зламаний рядок, подивись, що буде, і виправ.

In [ ]:
# Баг 1: sum = sum([540.0, 320.0])   ← розкоментуй разом з наступним рядком
# total_fares = sum([230, 180])
# BEGIN SOLUTION
total_bills = sum([540.0, 320.0])
total_fares = sum([230, 180])
# END SOLUTION
assert (total_bills, total_fares) == (860.0, 410)
print("✅ Баг 1: не називай змінні іменами вбудованих функцій")

In [ ]:
code_text = "summer10"
is_upper = code_text.isupper       # ← чому це «правда» для будь-якого рядка?
# BEGIN SOLUTION
is_upper = code_text.isupper()
# END SOLUTION
assert is_upper is False
print("✅ Баг 2: функція без дужок — об'єкт, і він завжди істинний")

In [ ]:
districts = ["Оболонь", "Поділ", "Печерськ"]
# print(sorted(districts, key=len(districts)))   ← TypeError: 'int' object is not callable
# BEGIN SOLUTION
result = sorted(districts, key=len)
# END SOLUTION
assert result == ["Поділ", "Оболонь", "Печерськ"]
print("✅ Баг 3: key= — функція, а не її результат")

## 🚀 Міні-проєкт: конвеєр для коментаря до замовлення

Клієнти пишуть коментарі до замовлень як завгодно: `"   без ЦИБУЛІ!!!   дзвонити   в домофон "`. Кухні й водієві потрібен акуратний текст. Кожен крок — чиста функція `str -> str`, конвеєр — список функцій, `run_pipeline` — той самий, що вище.

1. `strip_spaces` — прибирає пробіли по краях і зводить підряд кілька пробілів до одного;
2. `calm_down` — `"!!!"` → `"!"` (кілька знаків оклику — один);
3. `sentence_case` — перша літера велика, решта — малі;
4. ⭐ `make_limit(n)` — фабрика: крок, що обрізає текст до `n` символів і додає `"…"`, якщо обрізав.

In [ ]:
def strip_spaces(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return " ".join(text.split())
    # END SOLUTION


def calm_down(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    while "!!" in text:
        text = text.replace("!!", "!")
    return text
    # END SOLUTION


def sentence_case(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return text[:1].upper() + text[1:].lower()
    # END SOLUTION


def make_limit(n):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def limit(text):
        return text if len(text) <= n else text[:n] + "…"
    return limit
    # END SOLUTION


comment_pipeline = [strip_spaces, calm_down, sentence_case, make_limit(30)]
raw = "   без ЦИБУЛІ!!!   дзвонити   в домофон "
clean = run_pipeline(raw, comment_pipeline)
print(repr(clean))
assert run_pipeline(raw, comment_pipeline[:3]) == "Без цибулі! дзвонити в домофон"
assert clean == "Без цибулі! дзвонити в домофон"
assert make_limit(10)("Без цибулі! дзвонити") == "Без цибулі…"
assert run_pipeline("", comment_pipeline) == ""
print("✅ Міні-проєкт пройдено")

## ✅ Самоперевірка

1. Чим `fee_for` відрізняється від `fee_for("Поділ")`?
2. Що дає словник команд порівняно з ланцюжком `if`?
3. Чому `summer10` пам'ятає `percent`, хоча `make_discount` вже завершилась?
4. Навіщо `nonlocal`?
5. Чому порядок кроків у конвеєрі змінює ціну?

<details>
<summary>Відповіді</summary>

1. Об'єкт-функція проти результату виклику.
2. Нова команда — рядок у словнику; `run` не змінюється.
3. Це замикання: функція зберігає змінні оточення.
4. Щоб **змінити** змінну оточуючої функції; без нього присвоєння створює локальну.
5. Кожен крок працює з результатом попереднього.

</details>

### Шпаргалка

```python
handler = COMMANDS.get(name)            # словник команд
sorted(items, key=lambda d: d.fare)     # функція як аргумент
dispatch(delivery, notify_sms)          # callback — без дужок!

def make_discount(percent):             # фабрика + замикання
    def apply(bill):
        return bill * (100 - percent) / 100
    return apply

def make_counter():
    count = 0
    def next_id():
        nonlocal count                  # змінити змінну оточення
        count += 1
        return count
    return next_id

for step in steps:                      # конвеєр
    value = step(value)
```

## Далі

**Урок 19 — Класи, простір імен.** Лічильник `make_counter` — дані й функція разом. Клас зробить це явно.

Поглиблення — п'ять патернів у файлах `pattern_01_callback.md` … `pattern_05_pure_functions.md` у теці уроку.